In [ ]:
from ase import Atoms 
from ase.io import read 
from openbabel import openbabel as ob 
from monty.serialization import loadfn 
from collections import Counter,defaultdict
from tqdm.auto import tqdm 
import pandas as pd 
import glob 

### 1. Take an individual ase.atoms.Atoms object and determine if there are multiple molecules present

In [ ]:
def seperate_atoms(atoms: Atoms) -> list[Atoms]:
    """Split an ase.Atoms object into its disconnected molecular fragments."""
    mol = ob.OBMol()
    for Z, pos in zip(atoms.numbers, atoms.positions):
        a = mol.NewAtom()
        a.SetAtomicNum(int(Z))
        a.SetVector(*map(float, pos))
    mol.ConnectTheDots()  # guess bonds from interatomic distances

    return [
        Atoms(numbers=[a.GetAtomicNum() for a in ob.OBMolAtomIter(frag)],
              positions=[(a.GetX(), a.GetY(), a.GetZ()) for a in ob.OBMolAtomIter(frag)])
        for frag in mol.Separate()
    ]

In [292]:
combination = "combination_7"

In [293]:
atoms = read(f"{combination}/0.xtb.trj",format="xyz",index=0) # xtb trj file which is an xyz file 

In [294]:
seperate_atoms(atoms)

[Atoms(symbols='NO', pbc=False), Atoms(symbols='O2', pbc=False)]

### 2. generate some SMILES from the fragments 

In [295]:
from ase import Atoms
from openbabel import openbabel as ob

def to_obmol(atoms: Atoms) -> ob.OBMol:
    mol = ob.OBMol()
    for Z, pos in zip(atoms.numbers, atoms.positions):
        a = mol.NewAtom()
        a.SetAtomicNum(int(Z))
        a.SetVector(*map(float, pos))
    mol.ConnectTheDots()
    mol.PerceiveBondOrders()  # needed for correct double/aromatic bonds
    return mol

def fragment_smiles(atoms: Atoms, fmt: str = "can") -> list[str]:
    """Canonical SMILES for each disconnected fragment."""
    conv = ob.OBConversion()
    conv.SetOutFormat(fmt)
    return [conv.WriteString(frag).split()[0] for frag in to_obmol(atoms).Separate()]

In [296]:
fragments = seperate_atoms(atoms)
fragment_smiles(fragments[0]),fragment_smiles(fragments[1])

(['[N][O]'], ['O=O'])

### 3. Read in all frames of the XTB MD run and find new compounds 

In [297]:
frames =read(f"{combination}/0.xtb.trj",format="xyz",index=":")
len(frames)

2000

In [298]:
# read in the metadat which tells you the "reactants"

metadata = loadfn(f"{combination}/combination.json")
reactants = list(metadata.values())[0]

reactants

['NO', 'O2']

In [299]:
# run seperate_atoms for all frames and document unique structures and their appearance 
unique_structures = defaultdict(list)

for i,frame in tqdm(enumerate(frames),total=len(frames)):
    atoms_list = seperate_atoms(frame)
    for at in atoms_list:
        unique_structures["compositions"].append(at.get_chemical_formula()) 
        unique_structures["atoms"].append(at) 
        unique_structures["smiles"].append(fragment_smiles(at)[0])

Counter(unique_structures["compositions"])

  0%|          | 0/2000 [00:00<?, ?it/s]

Counter({'NO': 1341, 'O2': 1341, 'NO3': 659})

In [300]:
Counter(unique_structures["smiles"])
# as each is an unrelaxed snapshot, we only have 

Counter({'[N]=O': 1277,
         '[O][O]': 865,
         'O=O': 476,
         '[O]ON=O': 331,
         '[O]O[N][O]': 320,
         '[N][O]': 64,
         '[O]N1OO1': 8})

In [301]:
df = pd.DataFrame(unique_structures)

### 4 relax all fragmented structures 

In [ ]:
from tblite.ase import TBLite
from ase.optimize import BFGS

# uses the same as XTB 
def make_calc(frag):
    mult = 2 if sum(frag.numbers) % 2 else 1
    return TBLite(method="GFN2-xTB", multiplicity=mult, verbosity=0)

In [ ]:
relaxed_dict = defaultdict(list)
for frame in tqdm(df.T,total=len(df)):
    atoms = df.T[frame]["atoms"]
    atoms.calc = make_calc(atoms)
    BFGS(atoms, logfile=None).run(fmax=0.1, steps=200)
    relaxed_dict["relaxed_atoms"].append(atoms)
    relaxed_dict["relaxed_smiles"].append(fragment_smiles(atoms)[0])

In [ ]:
df["relaxed_smiles"] = relaxed_dict["relaxed_smiles"]
df["relaxed_atoms"] = relaxed_dict["relaxed_atoms"]

Counter(df["relaxed_smiles"])
#df[df["relaxed_smiles"] == '[O]N1OO1'] 

### 4. Alternatively symmetrise the molecule 

In [302]:
from pymatgen.io.ase import AseAtomsAdaptor
from pymatgen.symmetry.analyzer import PointGroupAnalyzer, iterative_symmetrize

def symmetrize(atoms: Atoms, tolerance: float = 0.3) -> tuple[Atoms, str]:
    """Return a symmetrised copy of a molecule and its point group."""
    mol = AseAtomsAdaptor.get_molecule(atoms, charge_spin_check=False)
    sym_mol = iterative_symmetrize(mol, max_n=10, tolerance=tolerance)["sym_mol"]
    pg = PointGroupAnalyzer(sym_mol, tolerance=tolerance).sch_symbol
    return AseAtomsAdaptor.get_atoms(sym_mol),pg 

In [303]:
sym,pg = symmetrize(fragments[0])
sym,pg

(MSONAtoms(symbols='NO', pbc=False), 'C*v')

In [305]:
symmetrised_dict = defaultdict(list)
for frame in tqdm(df.T,total=len(df)):
    atoms = df.T[frame]["atoms"]
    try:
        sym,pg = symmetrize(atoms)
    except:
        pass
    symmetrised_dict["symmetrised_atoms"].append(sym)
    symmetrised_dict["symmetrised_smiles"].append(fragment_smiles(sym)[0])
    symmetrised_dict["pg"].append(pg)

  0%|          | 0/3341 [00:00<?, ?it/s]

In [306]:
df["symmetrised_smiles"] = symmetrised_dict["symmetrised_smiles"]
df["symmetrised_atoms"] = symmetrised_dict["symmetrised_atoms"]
df["point_groups"] = symmetrised_dict["pg"]

Counter(df["symmetrised_smiles"])
#df[df["relaxed_smiles"] == '[O]N1OO1'] 

Counter({'[N]=O': 1278,
         '[O][O]': 865,
         'O=O': 476,
         '[O]ON=O': 330,
         '[O]O[N][O]': 320,
         '[N][O]': 64,
         '[O]N1OO1': 8})

In [ ]:
Counter(df["point_groups"])

## Building a Class for Kinetics 

In [ ]:
from ase import Atoms
from ase.io import read
from openbabel import openbabel as ob
from monty.serialization import loadfn
from collections import Counter, defaultdict
from python_utils import raise_exception
from tqdm.auto import tqdm
from pymatgen.io.ase import AseAtomsAdaptor
from pymatgen.symmetry.analyzer import PointGroupAnalyzer, iterative_symmetrize
import warnings
from tblite.ase import TBLite
from ase.optimize import BFGS

# uses the same as XTB


class AnalyseNanoReactor:
    def __init__(self, reactants: list = None,skip_step:int = 5):
        warnings.simplefilter("ignore")
        self.trajectory = defaultdict(dict)
        self.unique_structures = defaultdict(lambda: defaultdict(list))
        self.skip_step = skip_step
        if reactants:
            self.reactants = reactants

    def read_trajectory(self, trajectory: list | str):
        "reads one trajectory of multiple trajectories in xyz format (or xtb's .trj format )"
        if isinstance(trajectory, list):
            for i, traj in enumerate(trajectory):
                self.trajectory[i] = read(traj, format="xyz", index=f"::{self.skip_step}")
        elif isinstance(trajectory, str):
            self.trajectory[0] = read(trajectory, format="xyz", index=f"::{self.skip_step}")

    def relax_atoms(self, atoms: Atoms, fmax=0.2, steps=200):
        mult = 2 if sum(atoms.numbers) % 2 else 1
        atoms.calc = TBLite(method="GFN2-xTB", multiplicity=mult, verbosity=0)
        BFGS(atoms, logfile=None).run(fmax=0.1, steps=200)
        return atoms

    def seperate_atoms(self, atoms: Atoms) -> list[Atoms]:
        """Split an ase.Atoms object into its disconnected molecular fragments."""
        mol = ob.OBMol()
        for Z, pos in zip(atoms.numbers, atoms.positions):
            a = mol.NewAtom()
            a.SetAtomicNum(int(Z))
            a.SetVector(*map(float, pos))
        mol.ConnectTheDots()  # guess bonds from interatomic distances

        return [
            Atoms(numbers=[a.GetAtomicNum() for a in ob.OBMolAtomIter(frag)],
                  positions=[(a.GetX(), a.GetY(), a.GetZ()) for a in ob.OBMolAtomIter(frag)])
            for frag in mol.Separate()
        ]

    def to_obmol(self, atoms: Atoms) -> ob.OBMol:
        mol = ob.OBMol()
        for Z, pos in zip(atoms.numbers, atoms.positions):
            a = mol.NewAtom()
            a.SetAtomicNum(int(Z))
            a.SetVector(*map(float, pos))
        mol.ConnectTheDots()
        mol.PerceiveBondOrders()  # needed for correct double/aromatic bonds
        return mol

    def fragment_smiles(self, atoms: Atoms, fmt: str = "can") -> list[str]:
        """Canonical SMILES for each disconnected fragment."""
        conv = ob.OBConversion()
        conv.SetOutFormat(fmt)
        return [conv.WriteString(frag).split()[0] for frag in self.to_obmol(atoms).Separate()]

    def symmetrize(self, atoms: Atoms, tolerance: float = 0.3) -> tuple[Atoms, str]:
        """Return a symmetrised copy of a molecule and its point group."""
        mol = AseAtomsAdaptor.get_molecule(atoms, charge_spin_check=False)
        sym_mol = iterative_symmetrize(
            mol, max_n=10, tolerance=tolerance)["sym_mol"]
        pg = PointGroupAnalyzer(sym_mol, tolerance=tolerance).sch_symbol
        return AseAtomsAdaptor.get_atoms(sym_mol), pg

    def analyse_trajectory(self, index: int = 0, tolerance: float = 0.3, tqdm_kws={"disable": False}):
        for i, frame in tqdm(enumerate(self.trajectory[index]), total=len(self.trajectory[index]), **tqdm_kws):
            atoms_list = self.seperate_atoms(frame)
            for at in atoms_list:
                try:
                    at = self.relax_atoms(at)
                    sym, pg = self.symmetrize(at, tolerance=tolerance)
                    frags = self.fragment_smiles(sym)
                except Exception:
                    pass
                if len(frags) == 1:
                    self.unique_structures[index]["snapshot_atoms"].append(at)
                    self.unique_structures[index]["symmetrised_atoms"].append(
                        sym)
                    self.unique_structures[index]["compositions"].append(
                        sym.get_chemical_formula())
                    self.unique_structures[index]["smiles"].append(frags[0])
                    self.unique_structures[index]["point_group"].append(pg)

        self.unique_smiles_counter[index] = Counter(
            self.unique_structures[index]["smiles"])
        self.unique_compositions_counter[index] = Counter(
            self.unique_structures[index]["compositions"])
        self.unique_point_group_counter[index] = Counter(
            self.unique_structures[index]["point_group"])

        if self.reactants:
            indexes_a = [i for i, v in enumerate(
                self.unique_structures[0]["compositions"]) if v == self.reactants[0]]
            indexes_b = [i for i, v in enumerate(
                self.unique_structures[0]["compositions"]) if v == self.reactants[1]]
            indexes_to_delete = indexes_a + indexes_b

            df = pd.DataFrame(self.unique_structures[0])

            df = df.drop(index=indexes_to_delete)
            self.unique_structures[index]["snapshot_atoms"] = df["snapshot_atoms"].values
            self.unique_structures[index]["symmetrised_atoms"] = df["symmetrised_atoms"].values
            self.unique_structures[index]["compositions"] = df["compositions"].values
            self.unique_structures[index]["smiles"] = df["smiles"].values
            self.unique_structures[index]["point_group"] = df["point_group"].values
            self.unique_smiles_counter[index] = Counter(
                self.unique_structures[index]["smiles"])
            self.unique_compositions_counter[index] = Counter(
                self.unique_structures[index]["compositions"])
            self.unique_point_group_counter[index] = Counter(
                self.unique_structures[index]["point_group"])

    def analyse_trajectories(self, tolerance: float = 0.3, tqdm_kws={"disable": False}):
        if len(self.trajectory) > 1:
            for ii in self.trajectory:
                self.analyse_trajectory(
                    index=ii, tolerance=tolerance, tqdm_kws=tqdm_kws)
        else:
            print("Only one trajectory present, correct usage is AnalyseNanoReactor().analyse_trajectory. Running now...")
            self.analyse_trajectory(tolerance=tolerance, tqdm_kws=tqdm_kws)

In [335]:
anr = AnalyseNanoReactor(skip_step=5,reactants = loadfn("combination_7/combination.json")[str(7)])
anr.read_trajectory(["combination_7/0.xtb.trj","combination_7/1.xtb.trj"])

In [336]:
anr.analyse_trajectory()

  0%|          | 0/400 [00:00<?, ?it/s]

AttributeError: 'AnalyseNanoReactor' object has no attribute 'unique_smiles_counter'

In [ ]:
anr.unique_smiles_counter

Counter({'[O]ON=O': 647, '[O]N1OO1': 12})

In [322]:
unique_compositions_counter = Counter(anr.unique_structures[0]['compositions'])
list(unique_compositions_counter)

['NO3']

In [324]:
new_compounds = anr.reactants + list(unique_compositions_counter)
new_compounds

['NO', 'O2', 'NO3']

In [325]:
from reactit.reactit import ReactionGenerator 

In [329]:
rg = ReactionGenerator(new_compounds+["NO2"])

In [330]:
rg.iterate(3)

  0%|          | 0/12 [00:00<?, ?it/s]

['1 NO + 1 NO3 = 2 NO2',
 '2 NO + 1 O2 = 2 NO2',
 '1 NO + 1 O2 = 1 NO3',
 '1 O2 + 2 NO2 = 2 NO3']